In [25]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from ITIS_Model_funcs import get_nominal_param,OLS_res,ITIS

In [26]:
param_log,IC = get_nominal_param()
IC = [ 0.,          0.,          0.14759607,  0.,         13.75484703,  3.13344442, 15.73028492,  1.38649737]

D = '1'         # '1' for ACTH + Cort,
                # '2' for ACTH + Cort + TNF-a
                # '3' for ACTH + Cort + TNF-a + IL10
dpoints = '1'   # '1' for 25, '2' for 13

with open('syntheticData\\TRUE_SOL' + D + dpoints + '.pkl', 'rb') as f:
    results = pickle.load(f)

if D == '1':
    output_ids = [6,7]
elif D == '2':
    output_ids = [3,6,7]
else:
    output_ids = [3,4,6,7]



t_data = results['t_data']
y_data = results['y_data']

##SENSITIVITY ANALYSIS
h = 1e-6  #amount to perturb parameters
n_param = len(param_log)
n_states = len(output_ids)

S = np.zeros((n_param, len(t_data) * n_states)) ##Initialize shape of sensitivity matrix.

for i in range(n_param):  #calculate the relative residual sensitivity to each 45 parameters

        param_in = param_log[i]
        param_delta = param_in + h
##RIGHT NOW THIS IS BEING CALCULATED WITH NOISELESS DATA
        S[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_log, IC)
                                            - OLS_res(param_in, y_data, t_data, i, output_ids, param_log, IC)))

In [27]:
##import rankings from global analysis
with open('paramRankings\\rankingDesign' + D + '.pkl', 'rb') as f:
    results = pickle.load(f)

rank_value = results['rank_value']      # sorted ranking values
param_sorted = results['param_sorted']  #accordingly sorted params as strings
rank_cut = np.array(rank_value)[np.array(rank_value) > .25 * np.array(rank_value)[0]]
param_sorted = np.array(param_sorted)[:len(rank_cut)]

param_titles = ['d1',
'k1','k2','h1','h2','h3','d2',
'k3','k4','h4','d3',
'h5','h6','k5','k6','h7','d4',
'b1','k7','h8','k8','h9','d5','h10',
'b2','k9','k10','k11','d6',
'k12','k13','k14','h11','d7',
'k15','k16','d8',
'alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc']

circadian_param = ['alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc'] #exclude?

cond = 0
unid = []
p_indices = [param_titles.index(param_sorted[0])]
for i in range(1,len(rank_cut)):
    p_indices.append(param_titles.index(param_sorted[i]))
    S_opt = S[p_indices,:].copy()
    F_opt = S_opt@S_opt.T
    cond = np.linalg.cond(F_opt)
    if cond > 1e+5:
        unid.append(p_indices.pop())
        print("Removed: ", unid[-1])


S_opt = S[p_indices,:].copy()
F_opt = S_opt@S_opt.T
C_opt = np.linalg.inv(F_opt)
selected = [param_titles[i] for i in p_indices]
print("Design : ", D + dpoints)
print("Number of selected parameters:", len(p_indices))
print("Selected Parameters: ",  selected)
print(p_indices)
print("Excluded Parameters: ", [param_titles[i] for i in unid])
print("Condition Number of F:", np.linalg.cond(F_opt))
print("Diagonals of C:", np.diag(C_opt))

Design :  11
Number of selected parameters: 15
Selected Parameters:  ['d7', 'h6', 'd8', 'T', 'h11', 'beta', 'k3', 'd4', 'k14', 'h7', 'alpha', 'h4', 'k4', 'd6', 'k6']
[33, 12, 36, 43, 32, 39, 7, 16, 31, 15, 37, 9, 8, 28, 14]
Excluded Parameters:  []
Condition Number of F: 3007.608638903999
Diagonals of C: [0.08767256 0.10604571 0.02796229 0.05508357 0.11928236 0.16323879
 0.17100971 0.03700419 0.07138861 0.07709093 0.0413391  0.27950776
 0.07406219 0.05129255 0.2598072 ]


In [28]:
#Complete F
F = S@S.T
C = np.linalg.inv(F)
print("Condition Number of F:", np.linalg.cond(F))
print("Diagonals of C:", np.diag(C))
diag = np.diag(C)
unid = [param_titles[i] for i in list(np.where(diag > 1e+3))[0]]
print("Unidentifiable:", unid)

Condition Number of F: 11288994.21038686
Diagonals of C: [ 58.21422211 118.09745907  20.47986591  39.60312388  23.84630956
  38.63407056  61.98526861 131.05252028  21.12442513 106.10284807
  25.10245686  23.95363802  44.58091532  13.45261795  74.61894539
   8.15549457 137.10660343   7.53126134 160.87539616  77.95409923
  54.15416831 199.81859081  20.50949699 113.88817135  16.58753461
  36.21801833   1.37069841  47.91156666  12.35118128 148.74816645
   3.40600922   1.4072668   28.70172013  17.17782329   9.5306602
  28.50237612   3.2884959   81.63372674  29.79849075  52.71703276
  81.76716488  24.94943986  66.11795724   2.09454772 111.50476359]
Unidentifiable: []
